# V4a — VLM Names the Synthetic Confound (Colab)

Runs an **open-weight VLM (Qwen2-VL-7B-Instruct, 4-bit)** on the saliency crops exported
locally, using a **blind** prompt (never told what to look for). Writes `predictions.jsonl`
which you download and score locally with `python -m auditor.discovery score`.

**Runtime → Change runtime type → T4 GPU** before running.

**Inputs you upload:** `crops/` (folder of PNGs) and `manifest.json` from the local
`export` step (in the run's artifacts). **Output:** `predictions.jsonl`.

In [ ]:
# 1. Install dependencies (a few minutes on first run).
!pip -q install "transformers>=4.45" accelerate bitsandbytes qwen-vl-utils pillow

In [ ]:
# 2. Upload the crops folder (as crops.zip) and manifest.json.
#    Locally: zip the exported crops dir  ->  crops.zip, then upload both here.
from google.colab import files
import zipfile, os

print('Upload crops.zip and manifest.json ...')
up = files.upload()
if 'crops.zip' in up:
    with zipfile.ZipFile('crops.zip') as z:
        z.extractall('.')
# Expect a ./crops directory and ./manifest.json
assert os.path.isdir('crops'), 'crops/ folder not found after unzip'
assert os.path.exists('manifest.json'), 'manifest.json not found'
print('crops:', len(os.listdir('crops')), '| manifest present')

In [ ]:
# 3. Load Qwen2-VL-7B-Instruct in 4-bit (fits a free T4 ~15GB).
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2-VL-7B-Instruct'
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_quant_type='nf4')
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map='auto', torch_dtype=torch.float16)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print('model loaded')

In [ ]:
# 4. BLIND prompt — describes the task without naming tag/box/corner/marker.
PROMPT = (
    'This is a small region that a diagnostic model focused on within a chest X-ray. '
    'Describe anything artificial, non-anatomical, or out of place that you see in this region. '
    'If everything looks like normal anatomy, say so. Be specific and concise.'
)

def describe(image):
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image},
        {'type': 'text', 'text': PROMPT},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=80, do_sample=False)
    trimmed = out[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()

In [ ]:
# 5. Run over every crop in the manifest, write predictions.jsonl.
import json
from PIL import Image

manifest = json.load(open('manifest.json'))
with open('predictions.jsonl', 'w') as f:
    for i, row in enumerate(manifest):
        cid = row['id']
        img = Image.open(f'crops/{cid}.png').convert('RGB')
        desc = describe(img)
        f.write(json.dumps({'id': cid, 'description': desc}) + '\n')
        if i % 20 == 0:
            print(f'{i}/{len(manifest)}  {cid}: {desc[:70]}')
print('DONE ->', len(manifest), 'predictions')

In [ ]:
# 6. Download predictions.jsonl -> score locally with:
#    python -m auditor.discovery score --manifest manifest.json --predictions predictions.jsonl
from google.colab import files
files.download('predictions.jsonl')